In [1]:
import io
import chess.pgn
import pandas as pd
import numpy as np
import requests
import zstandard 
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import cross_val_score, cross_val_predict, RepeatedStratifiedKFold, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.base import clone, BaseEstimator, TransformerMixin
from stockfish import Stockfish

df = pd.read_csv('games_2017-08.csv')

In [2]:
stockfish = Stockfish(path=r"C:\Users\yazid\Downloads\stockfish-windows-x86-64-universal\stockfish\stockfish-windows-x86-64-universal.exe", depth=12, parameters={"Threads": 2})

In [3]:
def evaluate_position(stockfish, board):
    if board.is_checkmate():
        return -10000 if board.turn == chess.WHITE else 10000
    if board.is_game_over():
        return 0
    stockfish.set_fen_position(board.fen())
    result = stockfish.get_evaluation()
    if result['type'] == 'cp':
        value = result['value']
    else:
        value = 10000 if result['value'] > 0 else -10000
    return value if board.turn == chess.WHITE else -value

In [4]:
RANDOM_STATE = 42
CV = 5

In [ ]:
PIECE_VALUE = {
    chess.PAWN: 1, chess.KNIGHT: 3, chess.BISHOP: 3, chess.ROOK: 5, chess.QUEEN: 9
}

def material(board, color):
    return sum(PIECE_VALUE.get(p.piece_type, 0)
        for p in board.piece_map().values() if p.color == color)

def extract_features(stockfish, game, n_moves=10):
    h = game.headers
    base, inc = h['TimeControl'].split('+')
    board = game.board()
    clocks = {chess.WHITE: float('nan'), chess.BLACK: float('nan')}
    plies = 0
    last_node = None
    eval_cp = float('nan')
    for node in game.mainline():
        mover = board.turn
        board.push(node.move)
        c = node.clock()
        if c is not None:
            clocks[mover] = c
        plies += 1
        last_node = node
        if plies == 2 * n_moves:
            eval_cp = evaluate_position(stockfish, board)
            break
    if plies < 2 * n_moves:
        return None
    if last_node.is_end():
        return None
    if board.is_game_over():
        return None
    w_elo, b_elo = int(h['WhiteElo']), int(h['BlackElo'])
    return {
        'game_id': h['Site'].split('/')[-1],
        'date': h['UTCDate'], 'time': h['UTCTime'],
        'white': h['White'], 'black': h['Black'], 
        'white_elo': w_elo, 'black_elo': b_elo,
        'elo_diff': w_elo - b_elo,
        'base_sec': int(base), 'inc_sec': int(inc),
        'material_diff': material(board, chess.WHITE) - material(board, chess.BLACK), 'white_clock': clocks[chess.WHITE],
        'black_clock': clocks[chess.BLACK], 'clock_diff': clocks[chess.WHITE] - clocks[chess.BLACK],
        'fen': board.fen(),
        'eval_white': eval_cp,
        'result': h['Result']
    }

def read_games(stockfish, url, max_games=100, n_moves=10):
    resp = requests.get(url, stream=True)
    rows = []
    stream = io.TextIOWrapper(zstandard.ZstdDecompressor().stream_reader(resp.raw), encoding='utf-8')
    while len(rows) < max_games:
        game = chess.pgn.read_game(stream)
        if game is None:
            break
        try:
            features = extract_features(stockfish, game, n_moves)
            if features is not None:
                rows.append(features)
        except (KeyError, ValueError):
            continue
    resp.close()
    return pd.DataFrame(rows)
        

In [9]:
df = df[df['result'].isin(['1-0', '0-1'])].copy()
df['white_won'] = (df['result'] == '1-0').astype(int)

In [10]:
df.to_csv('games_2017-08.csv', index=False)
df = df.sort_values(by=['date', 'time']).reset_index(drop=True)

In [11]:
cut = int(len(df) * 0.8)
df_train = df.iloc[:cut].copy()
df_test = df.iloc[cut:].copy()

In [16]:
X_train = df_train.drop(columns=['game_id', 'result', 'white_won', 'date', 'time', 'white', 'black', 'fen'])
y_train = df_train['white_won']
X_test = df_test.drop(columns=['game_id', 'result', 'white_won', 'date', 'time', 'white', 'black', 'fen'])
test_id = df_test['game_id']

In [ ]:
COLUMN_TYPE = {'white_elo': 'num', 'black_elo': 'num', 'elo_diff': 'num', 'base_sec': 'num', 'inc_sec': 'num', 'material_diff': 'num',
               'white_clock': 'num', 'black_clock': 'num', 'clock_diff': 'num', 'eval_white': 'num'}


In [25]:
def split_by_type(columns):
    result = {'num': []}
    for col in columns:
        t = COLUMN_TYPE[col]
        result[t].append(col)
    return result

FEATURE_GROUPS = [
    ('core', ['white_elo', 'black_elo', 'elo_diff']), 
    ( '+clock', ['base_sec', 'inc_sec', 'white_clock', 'black_clock', 'clock_diff']),
    ( '+material', ['material_diff']),
    ('+position', ['eval_white'])
]